# BA870 Final Integrated Project Notebook

This notebook combines the final project pipeline into one clean workflow.


## Project Workflow Overview

This notebook implements an end-to-end financial analytics pipeline that transforms raw financial and market data into model-driven valuation insights and app-ready outputs.

---

### 1. Raw Data Inputs

We begin with two core external datasets:

- **Compustat fundamentals (`compustat_raw.csv`)**  
  Firm-level financial statement data (e.g., assets, earnings, revenue)

- **Fama-French factors (`f-f_research_data_factors.csv`)**  
  Market risk factors used for CAPM analysis

In addition, we dynamically pull **market data from yfinance** (prices, returns, volatility) for the selected healthcare ticker universe.

---

### 2. Panel Data Construction (`panel_data.csv`)

We merge Compustat fundamentals with market data at the **firm-year level** to create a unified panel dataset.

This dataset includes:
- Financial statement variables  
- Market-based features (returns, volatility)  
- Aligned ticker and time dimensions  

This serves as the **core analytical dataset** for downstream modeling.

---

### 3. Feature Engineering (`model_data.csv`)

From the panel data, we construct model-ready features, including:

- **Financial ratios** (e.g., profitability, leverage, valuation ratios)  
- **Growth metrics** (year-over-year changes)  
- **Relative features** (comparison to peer averages)  

The result is a cleaned and structured dataset used as input to the valuation model.

---

### 4. Model Development and Evaluation

#### Model Comparison (`valuation_model_comparison_results.csv`)

We train and compare multiple classification models to predict valuation categories, evaluating performance using:

- Accuracy  
- F1 Score  

This step ensures selection of an appropriate modeling approach.

#### Final Model Outputs

We implement a **logistic regression model** with a time-based train/test split.

Outputs include:

- **`valuation_test_predictions.csv`**  
  Predicted valuation classifications (overvalued / fairly valued / undervalued) and probabilities

- **`valuation_final_model_coefficients.csv`**  
  Model coefficients used for interpretability and feature importance

---

### 5. App-Specific Data Outputs

#### Valuation Inputs
- **`ticker_history_input.csv`**  
  Time-series inputs used in the valuation page of the app

#### Peer Comparison
- **`peer_summary.csv`**  
  Summarizes firm metrics relative to peers and supports benchmarking

#### Risk Analysis (CAPM)

We combine:
- Market data (monthly)
- Fama-French factors  

To compute:

- **`capm_dataset.csv`** (intermediate dataset)  
- **`capm_risk_metrics.csv`** (final output)

These include:
- Beta  
- Alpha  
- Volatility  
- Return metrics  

---

### 6. Final Outputs

The notebook produces all datasets required to power the application:

- Valuation predictions and model outputs  
- Peer comparison summaries  
- CAPM-based risk metrics  

Together, these outputs enable an integrated financial analysis workflow within the app.

## 0. Setup


In [ ]:
# SETUP
# Uncomment this in Colab if a package is missing.
!pip install yfinance statsmodels scikit-learn pandas numpy

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import statsmodels.api as sm

# All generated files are saved inside the Colab runtime.
# This avoids Google Drive path issues. Download or copy outputs to GitHub after running.
BASE_DIR = Path('/content/ba870_final_project')
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
APP_DIR = DATA_DIR / 'need for app'
MODELS_DIR = BASE_DIR / 'models'

for folder in [DATA_DIR, RAW_DIR, APP_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR.resolve())
print('DATA_DIR:', DATA_DIR.resolve())
print('RAW_DIR :', RAW_DIR.resolve())
print('APP_DIR :', APP_DIR.resolve())


BASE_DIR: /content/ba870_final_project
DATA_DIR: /content/ba870_final_project/data
RAW_DIR : /content/ba870_final_project/data/raw
APP_DIR : /content/ba870_final_project/data/need for app


## 1. Project constants


In [ ]:
# PROJECT CONSTANTS
# Healthcare ticker universe used throughout the project

tickers = [
    "LLY","JNJ","ABBV","MRK","UNH","AMGN","ABT","TMO","GILD","ISRG",
    "CVS","BMY","MDT","CI","ZTS","SYK","REGN","HCA","DHR","HUM",
    "VRTX","MRNA","PFE","BIIB","ILMN","EW","A","DXCM","IDXX","ALGN"
]

# Google Drive direct-download URLs from the original project notebooks
# These replace local loading for fixed external/raw datasets.
COMP_URL = "https://docs.google.com/uc?export=download&id=10t9uG_eU1f9w0b_p2f216mwM_3yfG3TH"
FF_RAW_URL = "https://docs.google.com/uc?export=download&id=1cEOlWOiUv_v1i-q2kIV2Pzu_z-h2CNgV"

# File names
raw_compustat_path = RAW_DIR / "compustat_raw.csv"
raw_ff_path = RAW_DIR / "f-f_research_data_factors.csv"

market_raw_path = RAW_DIR / "market_data_raw.csv"
market_monthly_raw_path = RAW_DIR / "market_data_raw_monthly.csv"

panel_path = DATA_DIR / "panel_data.csv"
model_data_path = DATA_DIR / "model_data.csv"
peer_summary_path = DATA_DIR / "peer_summary.csv"
ff_clean_path = DATA_DIR / "ff_factors_clean.csv"

capm_dataset_path = APP_DIR / "capm_dataset.csv"
capm_risk_metrics_path = DATA_DIR / "capm_risk_metrics.csv"

valuation_comparison_path = DATA_DIR / "valuation_model_comparison_results.csv"
valuation_predictions_path = DATA_DIR / "valuation_test_predictions.csv"
valuation_coefficients_path = DATA_DIR / "valuation_final_model_coefficients.csv"
ticker_history_path = DATA_DIR / "ticker_history_input.csv"

print('Output folder:', BASE_DIR)
print('Compustat source URL configured:', bool(COMP_URL))
print('Fama-French source URL configured:', bool(FF_RAW_URL))


Output folder: /content/ba870_final_project
Compustat source URL configured: True
Fama-French source URL configured: True


## 2. Helper functions


In [ ]:
# HELPER FUNCTIONS

def save_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print(f"Saved: {path} | shape={df.shape}")


def audit_csv(path):
    path = Path(path)
    if not path.exists():
        return {"file": str(path), "exists": False, "rows": None, "cols": None}
    df = pd.read_csv(path)
    return {"file": str(path), "exists": True, "rows": len(df), "cols": len(df.columns)}


def read_csv_from_url(url, save_path=None, **kwargs):
    if not url or "PASTE" in str(url):
        raise ValueError("Missing Google Drive URL. Paste the direct-download URL into the constants cell.")
    df = pd.read_csv(url, **kwargs)
    if save_path is not None:
        save_csv(df, save_path)
    return df


## 2A. Load fixed raw datasets from Google Drive URLs

In [ ]:
# DATA LOADING
# These files are fixed external/raw inputs, so we load them directly from Google Drive URLs.
# Market data is pulled fresh from yfinance in the next section.

compustat_raw = read_csv_from_url(COMP_URL, save_path=raw_compustat_path)
ff_raw_input = pd.read_csv(FF_RAW_URL, skiprows=3)

save_csv(ff_raw_input, raw_ff_path)

print("ff_raw_input shape:", ff_raw_input.shape)
ff_raw_input.head()
print('compustat_raw shape:', compustat_raw.shape)

Saved: /content/ba870_final_project/data/raw/compustat_raw.csv | shape=(387, 24)
Saved: /content/ba870_final_project/data/raw/f-f_research_data_factors.csv | shape=(1298, 5)
ff_raw_input shape: (1298, 5)
compustat_raw shape: (387, 24)


## 3. Pull fresh yfinance market data

This section always pulls fresh yearly and monthly market data from yfinance and saves it to `data/raw/`.


In [ ]:
# PULL YEARLY MARKET DATA
import yfinance as yf

data = yf.download(
    tickers,
    start='2013-01-01',
    end='2026-01-01',
    auto_adjust=True,
    progress=False
)

prices = data['Close']

yearly_prices = prices.stack().reset_index()
yearly_prices.columns = ['date', 'ticker', 'price']
yearly_prices = yearly_prices.sort_values(['ticker', 'date']).reset_index(drop=True)

yearly_prices['return'] = yearly_prices.groupby('ticker')['price'].pct_change()
yearly_prices['past_12m_return'] = yearly_prices.groupby('ticker')['price'].pct_change(252)
yearly_prices['future_12m_return'] = yearly_prices.groupby('ticker')['price'].pct_change(252).shift(-252)
yearly_prices['volatility'] = (
    yearly_prices.groupby('ticker')['return']
    .rolling(252)
    .std()
    .reset_index(level=0, drop=True)
)
yearly_prices['year'] = pd.to_datetime(yearly_prices['date']).dt.year

yearly_df = (
    yearly_prices.sort_values('date')
    .groupby(['ticker', 'year'])
    .tail(1)
    .reset_index(drop=True)
)

save_csv(yearly_df, market_raw_path)
yearly_df.head()


Saved: /content/ba870_final_project/data/raw/market_data_raw.csv | shape=(385, 8)


,date,ticker,price,return,past_12m_return,future_12m_return,volatility,year
0,2013-12-31,BMY,35.518055,0.001696,NaN,0.141800,NaN,2013
1,2013-12-31,ISRG,42.675556,0.002637,NaN,0.377161,NaN,2013
2,2013-12-31,ZTS,29.484987,0.000612,NaN,0.327979,NaN,2013
3,2013-12-31,GILD,51.881187,0.000266,NaN,0.255127,NaN,2013
4,2013-12-31,IDXX,53.185001,-0.002999,NaN,0.393908,NaN,2013


In [ ]:
# PULL MONTHLY MARKET DATA
import yfinance as yf

data = yf.download(
    tickers,
    start='2013-01-01',
    end='2026-01-01',
    auto_adjust=True,
    progress=False
)

prices = data['Close']

monthly_prices = prices.stack().reset_index()
monthly_prices.columns = ['date', 'ticker', 'price']
monthly_prices = monthly_prices.sort_values(['ticker', 'date']).reset_index(drop=True)
monthly_prices['month_end'] = pd.to_datetime(monthly_prices['date']) + pd.offsets.MonthEnd(0)

monthly_df = (
    monthly_prices.sort_values(['ticker', 'date'])
    .groupby(['ticker', 'month_end'])
    .tail(1)
    .copy()
)

monthly_df = monthly_df.rename(columns={'month_end': 'date_month_end'})
monthly_df = monthly_df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

monthly_df['return'] = monthly_df.groupby('ticker')['price'].pct_change()
monthly_df['past_12m_return'] = monthly_df.groupby('ticker')['price'].pct_change(12)
monthly_df['future_12m_return'] = monthly_df.groupby('ticker')['price'].pct_change(12).shift(-12)
monthly_df['volatility'] = (
    monthly_df.groupby('ticker')['return']
    .rolling(12)
    .std()
    .reset_index(level=0, drop=True)
)

monthly_df = monthly_df.drop(columns=['date'])
monthly_df = monthly_df.rename(columns={'date_month_end': 'date'})
monthly_df = monthly_df[[
    'date', 'ticker', 'price', 'return', 'past_12m_return', 'future_12m_return', 'volatility'
]].copy()
monthly_df = monthly_df.sort_values(['ticker', 'date']).reset_index(drop=True)

save_csv(monthly_df, market_monthly_raw_path)
monthly_df.head()


Saved: /content/ba870_final_project/data/raw/market_data_raw_monthly.csv | shape=(4608, 7)


,date,ticker,price,return,past_12m_return,future_12m_return,volatility
0,2013-01-31,A,28.626429,NaN,NaN,0.311951,NaN
1,2013-02-28,A,26.516850,-0.073693,NaN,0.386609,NaN
2,2013-03-31,A,26.906790,0.014705,NaN,0.342271,NaN
3,2013-04-30,A,26.567017,-0.012628,NaN,0.316915,NaN
4,2013-05-31,A,29.137810,0.096766,NaN,0.265160,NaN


## 4. Build panel data

This creates `panel_data.csv` from `compustat_raw.csv` and the fresh yfinance yearly data.


In [ ]:
market_data = pd.read_csv(market_raw_path)
compustat = compustat_raw.copy()

market_data["date"] = pd.to_datetime(market_data["date"])
compustat["datadate"] = pd.to_datetime(compustat["datadate"])

market_data["ticker"] = market_data["ticker"].astype(str).str.upper().str.strip()
compustat["tic"] = compustat["tic"].astype(str).str.upper().str.strip()

market_data["year"] = market_data["year"].astype(int)
compustat["year"] = compustat["fyear"].astype(int)

market_keep = [
    "ticker",
    "year",
    "date",
    "price",
    "return",
    "past_12m_return",
    "future_12m_return",
    "volatility"
]

compustat_keep = [
    "gvkey",
    "tic",
    "conm",
    "datadate",
    "fyear",
    "year",
    "sale",
    "ni",
    "at",
    "lt",
    "dltt",
    "dlc",
    "seq",
    "csho",
    "prcc_f",
    "act",
    "lct",
    "che",
    "oibdp",
    "capx"
]

market_data = market_data[market_keep].copy()
compustat = compustat[compustat_keep].copy()

market_data = market_data.drop_duplicates(subset=["ticker", "year"], keep="last").copy()
compustat = compustat.drop_duplicates(subset=["tic", "year"], keep="last").copy()

panel_data = pd.merge(
    compustat,
    market_data,
    left_on=["tic", "year"],
    right_on=["ticker", "year"],
    how="inner"
)

panel_columns = [
    "gvkey",
    "tic",
    "ticker",
    "conm",
    "fyear",
    "year",
    "datadate",
    "date",
    "sale",
    "ni",
    "at",
    "lt",
    "dltt",
    "dlc",
    "seq",
    "csho",
    "prcc_f",
    "act",
    "lct",
    "che",
    "oibdp",
    "capx",
    "price",
    "return",
    "past_12m_return",
    "future_12m_return",
    "volatility"
]

panel_data = panel_data[panel_columns].copy()
panel_data = panel_data.sort_values(["ticker", "year"]).reset_index(drop=True)
panel_data = panel_data.drop(columns=["tic", "fyear"])

save_csv(panel_data, panel_path)

print("panel_data shape:", panel_data.shape)
print("Unique firms:", panel_data["ticker"].nunique())
print("Year range:", panel_data["year"].min(), "to", panel_data["year"].max())
panel_data.head()


Saved: /content/ba870_final_project/data/panel_data.csv | shape=(384, 25)
panel_data shape: (384, 25)
Unique firms: 30
Year range: 2013 to 2025


,gvkey,ticker,conm,year,datadate,date,sale,ni,at,lt,...,act,lct,che,oibdp,capx,price,return,past_12m_return,future_12m_return,volatility
0,126554,A,AGILENT TECHNOLOGIES INC,2013,2013-10-31,2013-12-31,6782.0,724.0,10686.0,5397.0,...,4983.0,1602.0,2675.0,1441.0,195.0,36.936432,-0.002269,NaN,0.007828,NaN
1,126554,A,AGILENT TECHNOLOGIES INC,2014,2014-10-31,2014-12-31,6981.0,504.0,10831.0,5530.0,...,5500.0,1702.0,3028.0,1519.0,205.0,37.225578,-0.010394,0.007828,0.034650,0.013954
2,126554,A,AGILENT TECHNOLOGIES INC,2015,2015-10-31,2015-12-31,4038.0,401.0,7479.0,3309.0,...,3686.0,976.0,2245.0,859.0,98.0,38.515442,-0.005826,0.034650,0.101627,0.014177
3,126554,A,AGILENT TECHNOLOGIES INC,2016,2016-10-31,2016-12-30,4202.0,462.0,7802.0,3556.0,...,3635.0,945.0,2289.0,963.0,139.0,42.429649,-0.001753,0.101627,0.497132,0.014795
4,126554,A,AGILENT TECHNOLOGIES INC,2017,2017-10-31,2017-12-29,4472.0,684.0,8426.0,3591.0,...,4169.0,1263.0,2678.0,1095.0,176.0,62.930767,-0.004918,0.480579,-0.010021,0.010432


# 5. Build model-ready dataset

This creates `model_data.csv` with financial ratios, change features, and peer-relative features.


In [ ]:
panel_data = pd.read_csv(panel_path)

panel_data["datadate"] = pd.to_datetime(panel_data["datadate"])
panel_data["date"] = pd.to_datetime(panel_data["date"])
panel_data["ticker"] = panel_data["ticker"].astype(str).str.upper().str.strip()

panel_data = panel_data.sort_values(["ticker", "year"]).reset_index(drop=True)

# Base financial features
panel_data["roa"] = panel_data["ni"] / panel_data["at"]
panel_data["operating_margin"] = panel_data["oibdp"] / panel_data["sale"]
panel_data["debt_to_assets"] = (panel_data["dltt"] + panel_data["dlc"]) / panel_data["at"]
panel_data["revenue_growth"] = panel_data.groupby("ticker")["sale"].pct_change()
panel_data["current_ratio"] = panel_data["act"] / panel_data["lct"]
panel_data["log_assets"] = np.log(panel_data["at"])
panel_data["market_cap"] = panel_data["prcc_f"] * panel_data["csho"]
panel_data["price_to_sales"] = panel_data["market_cap"] / panel_data["sale"]
panel_data["price_to_book"] = panel_data["market_cap"] / panel_data["seq"]

# Change features
panel_data["roa_change"] = panel_data.groupby("ticker")["roa"].diff()
panel_data["operating_margin_change"] = panel_data.groupby("ticker")["operating_margin"].diff()
panel_data["debt_to_assets_change"] = panel_data.groupby("ticker")["debt_to_assets"].diff()

# Relative features versus yearly peer median
base_compare_cols = [
    "roa",
    "operating_margin",
    "debt_to_assets",
    "price_to_sales",
    "price_to_book"
]

yearly_medians = panel_data.groupby("year")[base_compare_cols].median().reset_index()
yearly_medians = yearly_medians.rename(columns={
    "roa": "roa_median",
    "operating_margin": "operating_margin_median",
    "debt_to_assets": "debt_to_assets_median",
    "price_to_sales": "price_to_sales_median",
    "price_to_book": "price_to_book_median"
})

panel_data = pd.merge(panel_data, yearly_medians, on="year", how="left")

panel_data["roa_rel"] = panel_data["roa"] - panel_data["roa_median"]
panel_data["operating_margin_rel"] = panel_data["operating_margin"] - panel_data["operating_margin_median"]
panel_data["debt_to_assets_rel"] = panel_data["debt_to_assets"] - panel_data["debt_to_assets_median"]
panel_data["price_to_sales_rel"] = panel_data["price_to_sales"] - panel_data["price_to_sales_median"]
panel_data["price_to_book_rel"] = panel_data["price_to_book"] - panel_data["price_to_book_median"]

model_columns = [
    "ticker",
    "gvkey",
    "year",
    "past_12m_return",
    "volatility",
    "roa",
    "operating_margin",
    "debt_to_assets",
    "revenue_growth",
    "current_ratio",
    "log_assets",
    "price_to_sales",
    "price_to_book",
    "roa_change",
    "operating_margin_change",
    "debt_to_assets_change",
    "roa_rel",
    "operating_margin_rel",
    "debt_to_assets_rel",
    "price_to_sales_rel",
    "price_to_book_rel",
    "market_cap"
]

model_data = panel_data[model_columns].copy()
model_data = model_data.replace([np.inf, -np.inf], np.nan)

winsor_cols = [
    "past_12m_return",
    "volatility",
    "roa",
    "operating_margin",
    "debt_to_assets",
    "revenue_growth",
    "current_ratio",
    "log_assets",
    "price_to_sales",
    "price_to_book",
    "roa_change",
    "operating_margin_change",
    "debt_to_assets_change",
    "roa_rel",
    "operating_margin_rel",
    "debt_to_assets_rel",
    "price_to_sales_rel",
    "price_to_book_rel"
]

for col in winsor_cols:
    lower = model_data[col].quantile(0.01)
    upper = model_data[col].quantile(0.99)
    model_data[col] = model_data[col].clip(lower=lower, upper=upper)

required_model_fields = [
    "roa",
    "operating_margin",
    "debt_to_assets",
    "revenue_growth",
    "current_ratio",
    "log_assets",
    "price_to_sales",
    "price_to_book",
    "roa_change",
    "operating_margin_change",
    "debt_to_assets_change",
    "roa_rel",
    "operating_margin_rel",
    "debt_to_assets_rel",
    "price_to_sales_rel",
    "price_to_book_rel"
]

model_data = model_data.dropna(subset=required_model_fields).copy()
model_data = model_data.sort_values(["ticker", "year"]).reset_index(drop=True)

save_csv(model_data, model_data_path)

print("model_data shape:", model_data.shape)
print("Unique firms:", model_data["ticker"].nunique())
print("Year range:", model_data["year"].min(), "to", model_data["year"].max())
model_data.head()


Saved: /content/ba870_final_project/data/model_data.csv | shape=(350, 22)
model_data shape: (350, 22)
Unique firms: 30
Year range: 2014 to 2025


,ticker,gvkey,year,past_12m_return,volatility,roa,operating_margin,debt_to_assets,revenue_growth,current_ratio,...,price_to_book,roa_change,operating_margin_change,debt_to_assets_change,roa_rel,operating_margin_rel,debt_to_assets_rel,price_to_sales_rel,price_to_book_rel,market_cap
0,A,126554,2014,0.007828,0.013954,0.046533,0.217591,0.255009,0.029342,3.231492,...,3.495077,-0.021219,0.005116,0.002435,-0.017866,-0.035511,0.038307,-1.825862,-1.203662,18516.92048
1,A,126554,2015,0.034650,0.014177,0.053617,0.212729,0.221286,-0.421573,3.776639,...,3.008476,0.007084,-0.004862,-0.033723,-0.014094,-0.042523,-0.039219,-0.992091,-1.064679,12536.32000
2,A,126554,2016,0.101627,0.014795,0.059216,0.229177,0.245065,0.040614,3.846561,...,3.321517,0.005599,0.016448,0.023779,-0.021381,-0.045790,-0.025802,-0.613293,-0.937162,14093.19577
3,A,126554,2017,0.480579,0.010432,0.081177,0.244857,0.238666,0.064255,3.300871,...,4.534042,0.021962,0.015680,-0.006399,0.011568,-0.028528,-0.015471,0.081396,-0.441615,21903.95925
4,A,126554,2018,0.011654,0.017025,0.036998,0.244404,0.210631,0.098837,3.286080,...,4.507282,-0.044179,-0.000453,-0.028035,-0.053746,-0.032538,-0.069150,-0.173983,-0.081242,20584.75485


## 6. Valuation model comparison

This creates `valuation_model_comparison_results.csv`.

This section uses the original model comparison logic and then transitions into the final logistic regression model.


In [ ]:
model_data = pd.read_csv(model_data_path)

# Original valuation score used for model comparison
cheapness = (-model_data["price_to_sales_rel"]) + (-model_data["price_to_book_rel"])
quality = model_data["roa_rel"] + model_data["operating_margin_rel"]
health = (-model_data["debt_to_assets_rel"])
growth = model_data["revenue_growth"]

model_data["valuation_score_comparison"] = (
    cheapness
    + quality
    + 0.5 * health
    + 0.5 * growth
)

low_cut = model_data["valuation_score_comparison"].quantile(0.30)
high_cut = model_data["valuation_score_comparison"].quantile(0.70)

def make_valuation_label_comparison(x):
    if x <= low_cut:
        return 0
    elif x >= high_cut:
        return 2
    else:
        return 1

model_data["valuation_label_comparison"] = model_data["valuation_score_comparison"].apply(make_valuation_label_comparison)

feature_cols_comparison = [
    "roa",
    "operating_margin",
    "debt_to_assets",
    "revenue_growth",
    "current_ratio",
    "log_assets",
    "price_to_sales",
    "price_to_book",
    "roa_change",
    "operating_margin_change",
    "debt_to_assets_change",
    "roa_rel",
    "operating_margin_rel",
    "debt_to_assets_rel",
    "price_to_sales_rel",
    "price_to_book_rel"
]

X = model_data[feature_cols_comparison].copy()
y = model_data["valuation_label_comparison"].copy()

train_mask = model_data["year"] <= 2020
test_mask = model_data["year"] >= 2021

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

def evaluate_multiclass_model(name, model, X_train, y_train, X_test, y_test):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    return {
        "model": name,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "train_f1": f1_score(y_train, train_pred, average="macro"),
        "test_f1": f1_score(y_test, test_pred, average="macro"),
        "train_pred": train_pred,
        "test_pred": test_pred
    }

# Logistic regression
logit_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000, random_state=42))
])

logit_grid = GridSearchCV(
    estimator=logit_pipe,
    param_grid={"model__C": [0.01, 0.1, 1, 10]},
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)
logit_grid.fit(X_train, y_train)
logit_eval = evaluate_multiclass_model(
    "Logistic Regression",
    logit_grid.best_estimator_,
    X_train, y_train, X_test, y_test
)

# Decision tree
dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid={"max_depth": [2, 3, 5, 8, None], "min_samples_leaf": [1, 3, 5, 10]},
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)
dt_eval = evaluate_multiclass_model(
    "Decision Tree",
    dt_grid.best_estimator_,
    X_train, y_train, X_test, y_test
)

# Random forest
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid={"n_estimators": [100, 200], "max_depth": [3, 5, 8, None], "min_samples_leaf": [1, 3, 5]},
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
rf_eval = evaluate_multiclass_model(
    "Random Forest",
    rf_grid.best_estimator_,
    X_train, y_train, X_test, y_test
)

# Gradient boosting
gbc_grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid={"n_estimators": [50, 100, 200], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [1, 2, 3]},
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)
gbc_grid.fit(X_train, y_train)
gbc_eval = evaluate_multiclass_model(
    "Gradient Boosting",
    gbc_grid.best_estimator_,
    X_train, y_train, X_test, y_test
)

valuation_model_comparison_results = pd.DataFrame([
    {k: logit_eval[k] for k in ["model", "train_accuracy", "test_accuracy", "train_f1", "test_f1"]},
    {k: dt_eval[k] for k in ["model", "train_accuracy", "test_accuracy", "train_f1", "test_f1"]},
    {k: rf_eval[k] for k in ["model", "train_accuracy", "test_accuracy", "train_f1", "test_f1"]},
    {k: gbc_eval[k] for k in ["model", "train_accuracy", "test_accuracy", "train_f1", "test_f1"]},
]).sort_values(["test_f1", "test_accuracy"], ascending=False).reset_index(drop=True)

save_csv(valuation_model_comparison_results, valuation_comparison_path)

print("Valuation Model Comparison:")
valuation_model_comparison_results


Saved: /content/ba870_final_project/data/valuation_model_comparison_results.csv | shape=(4, 5)
Valuation Model Comparison:


,model,train_accuracy,test_accuracy,train_f1,test_f1
0,Logistic Regression,0.965174,0.919463,0.965573,0.922025
1,Random Forest,0.990050,0.912752,0.990286,0.914400
2,Gradient Boosting,1.000000,0.906040,1.000000,0.908897
3,Decision Tree,0.965174,0.859060,0.965767,0.866003


## 7. Final valuation model: logistic regression

This creates:

- `valuation_test_predictions.csv`
- `valuation_final_model_coefficients.csv`
- `ticker_history_input.csv`

This is the final model used for the project.


In [ ]:
model_data = pd.read_csv(model_data_path)

# Revised final valuation score
cheapness = (-model_data["price_to_sales_rel"]) + (-model_data["price_to_book_rel"])
quality = model_data["roa_rel"] + model_data["operating_margin_rel"]
health = (-model_data["debt_to_assets_rel"])
growth = model_data["revenue_growth"]

model_data["valuation_score"] = (
    cheapness
    + 0.75 * quality
    + 0.50 * health
    + 0.25 * growth
)

low_cut = model_data["valuation_score"].quantile(0.30)
high_cut = model_data["valuation_score"].quantile(0.70)

def make_valuation_label(x):
    if x <= low_cut:
        return 0
    elif x >= high_cut:
        return 2
    else:
        return 1

model_data["valuation_label"] = model_data["valuation_score"].apply(make_valuation_label)

feature_cols = [
    "roa",
    "operating_margin",
    "debt_to_assets",
    "revenue_growth",
    "current_ratio",
    "log_assets",
    "price_to_sales",
    "price_to_book",
    "roa_change",
    "operating_margin_change",
    "debt_to_assets_change",
    "roa_rel",
    "operating_margin_rel",
    "debt_to_assets_rel",
    "price_to_sales_rel",
    "price_to_book_rel"
]

X = model_data[feature_cols].copy()
y = model_data["valuation_label"].copy()
id_cols = model_data[["ticker", "year", "valuation_score"]].copy()

train_mask = model_data["year"] <= 2020
test_mask = model_data["year"] >= 2021

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

id_test = id_cols.loc[test_mask].copy().reset_index(drop=True)

logit_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000, random_state=42))
])

logit_grid = GridSearchCV(
    estimator=logit_pipe,
    param_grid={"model__C": [0.01, 0.1, 1, 10]},
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

logit_grid.fit(X_train, y_train)
final_model = logit_grid.best_estimator_

train_pred = final_model.predict(X_train)
test_pred = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)
train_f1 = f1_score(y_train, train_pred, average="macro")
test_f1 = f1_score(y_test, test_pred, average="macro")

print("Final Logistic Regression Best Params:", logit_grid.best_params_)
print("Train Accuracy:", round(train_accuracy, 6))
print("Test Accuracy :", round(test_accuracy, 6))
print("Train F1      :", round(train_f1, 6))
print("Test F1       :", round(test_f1, 6))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

test_predictions = id_test.copy()
test_predictions["actual_valuation_label"] = y_test.values
test_predictions["predicted_valuation_label"] = test_pred

# Add class probabilities for app/explainability
for idx, class_label in enumerate(final_model.named_steps["model"].classes_):
    test_predictions[f"prob_class_{class_label}"] = test_proba[:, idx]

test_predictions["valuation_class_text"] = test_predictions["predicted_valuation_label"].map({
    0: "Overvalued",
    1: "Fairly Valued",
    2: "Undervalued"
})

save_csv(test_predictions, valuation_predictions_path)

coefs = pd.DataFrame({
    "feature": feature_cols,
    "coefficient_class_0": final_model.named_steps["model"].coef_[0],
    "coefficient_class_1": final_model.named_steps["model"].coef_[1],
    "coefficient_class_2": final_model.named_steps["model"].coef_[2]
})

save_csv(coefs, valuation_coefficients_path)

history_input_df = model_data[["ticker", "year"] + feature_cols].copy()
history_input_df = history_input_df.sort_values(["ticker", "year"]).reset_index(drop=True)

save_csv(history_input_df, ticker_history_path)

# Optional saved model artifacts
import pickle

with open(MODELS_DIR / "final_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open(MODELS_DIR / "feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols, f)

print("Saved model artifacts in:", MODELS_DIR)


Final Logistic Regression Best Params: {'model__C': 10}
Train Accuracy: 0.965174
Test Accuracy : 0.919463
Train F1      : 0.965685
Test F1       : 0.922083

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        40
           1       0.93      0.88      0.90        64
           2       0.88      0.98      0.93        45

    accuracy                           0.92       149
   macro avg       0.92      0.93      0.92       149
weighted avg       0.92      0.92      0.92       149


Confusion Matrix (Test):
[[37  3  0]
 [ 2 56  6]
 [ 0  1 44]]
Saved: /content/ba870_final_project/data/valuation_test_predictions.csv | shape=(149, 9)
Saved: /content/ba870_final_project/data/valuation_final_model_coefficients.csv | shape=(16, 4)
Saved: /content/ba870_final_project/data/ticker_history_input.csv | shape=(350, 18)
Saved model artifacts in: /content/ba870_final_project/models


## 8. Peer summary output

This creates `peer_summary.csv`.


In [ ]:
model_data = pd.read_csv(model_data_path)

features = [
    "roa",
    "operating_margin",
    "debt_to_assets",
    "revenue_growth",
    "current_ratio",
    "log_assets",
    "price_to_sales",
    "price_to_book",
    "market_cap"
]

peer_medians = model_data.groupby("year")[features].median().reset_index()
peer_medians = peer_medians.rename(columns={col: f"{col}_median" for col in features})

peer_summary = pd.merge(
    model_data,
    peer_medians,
    on="year",
    how="left"
)

for col in features:
    peer_summary[f"{col}_pct"] = peer_summary.groupby("year")[col].rank(pct=True)

keep_cols = [
    "ticker",
    "year",
    *features,
    *[f"{col}_median" for col in features],
    *[f"{col}_pct" for col in features]
]

peer_summary = peer_summary[keep_cols].copy()
peer_summary = peer_summary.sort_values(["ticker", "year"]).reset_index(drop=True)

save_csv(peer_summary, peer_summary_path)

print("peer_summary shape:", peer_summary.shape)
peer_summary.head()


Saved: /content/ba870_final_project/data/peer_summary.csv | shape=(350, 29)
peer_summary shape: (350, 29)


,ticker,year,roa,operating_margin,debt_to_assets,revenue_growth,current_ratio,log_assets,price_to_sales,price_to_book,...,market_cap_median,roa_pct,operating_margin_pct,debt_to_assets_pct,revenue_growth_pct,current_ratio_pct,log_assets_pct,price_to_sales_pct,price_to_book_pct,market_cap_pct
0,A,2014,0.046533,0.217591,0.255009,0.029342,3.231492,9.290168,2.652474,3.495077,...,55270.219650,0.214286,0.250000,0.642857,0.285714,0.714286,0.357143,0.178571,0.392857,0.178571
1,A,2015,0.053617,0.212729,0.221286,-0.421573,3.776639,8.919854,3.104586,3.008476,...,60300.393315,0.392857,0.285714,0.357143,0.035714,0.892857,0.321429,0.214286,0.285714,0.142857
2,A,2016,0.059216,0.229177,0.245065,0.040614,3.846561,8.962135,3.353926,3.321517,...,54698.180400,0.428571,0.250000,0.321429,0.321429,0.857143,0.357143,0.321429,0.357143,0.142857
3,A,2017,0.081177,0.244857,0.238666,0.064255,3.300871,9.039077,4.898023,4.534042,...,66017.983500,0.642857,0.250000,0.392857,0.357143,0.785714,0.285714,0.535714,0.428571,0.142857
4,A,2018,0.036998,0.244404,0.210631,0.098837,3.286080,9.052633,4.189002,4.507282,...,72338.680000,0.172414,0.275862,0.310345,0.551724,0.793103,0.275862,0.448276,0.448276,0.137931


## 9. Clean Fama-French factors

This creates `ff_factors_clean.csv` from `f-f_research_data_factors.csv`.


In [ ]:
# The raw Kenneth French file has descriptive rows before the monthly data.
ff_raw = read_csv_from_url(FF_RAW_URL, skiprows=3)

ff_raw = ff_raw.rename(columns={ff_raw.columns[0]: "date_key"})

ff_monthly = ff_raw.copy()
ff_monthly["date_key"] = ff_monthly["date_key"].astype(str).str.strip()
ff_monthly = ff_monthly[ff_monthly["date_key"].str.match(r"^\d{6}$")].copy()

ff_monthly.columns = ["date_key", "mkt_rf", "smb", "hml", "rf"]

for col in ["mkt_rf", "smb", "hml", "rf"]:
    ff_monthly[col] = pd.to_numeric(ff_monthly[col], errors="coerce") / 100.0

ff_monthly["date"] = pd.to_datetime(ff_monthly["date_key"], format="%Y%m") + pd.offsets.MonthEnd(0)

ff_factors_clean = ff_monthly[["date", "mkt_rf", "smb", "hml", "rf"]].copy()
ff_factors_clean = ff_factors_clean.sort_values("date").reset_index(drop=True)

save_csv(ff_factors_clean, ff_clean_path)

print("FF factor date range:", ff_factors_clean["date"].min(), "to", ff_factors_clean["date"].max())
ff_factors_clean.head()


Saved: /content/ba870_final_project/data/ff_factors_clean.csv | shape=(1196, 5)
FF factor date range: 1926-07-31 00:00:00 to 2026-02-28 00:00:00


,date,mkt_rf,smb,hml,rf
0,1926-07-31,0.0289,-0.0255,-0.0239,0.0022
1,1926-08-31,0.0264,-0.0114,0.0381,0.0025
2,1926-09-30,0.0038,-0.0136,0.0005,0.0023
3,1926-10-31,-0.0327,-0.0014,0.0082,0.0032
4,1926-11-30,0.0254,-0.0011,-0.0061,0.0031


## 10. Build CAPM dataset

This creates `data/need for app/capm_dataset.csv`.


In [ ]:
market_data = pd.read_csv(market_monthly_raw_path)
ff_factors = pd.read_csv(ff_clean_path)

market_data["date"] = pd.to_datetime(market_data["date"])
ff_factors["date"] = pd.to_datetime(ff_factors["date"])

market_data = market_data.sort_values(["ticker", "date"]).copy()

if "return" not in market_data.columns:
    market_data["return"] = market_data.groupby("ticker")["price"].pct_change()

monthly_returns = market_data[["date", "ticker", "return"]].copy()
monthly_returns = monthly_returns.rename(columns={"return": "stock_return"})

ff_factors_capm = ff_factors[["date", "mkt_rf", "rf"]].copy()

capm_dataset = pd.merge(
    monthly_returns,
    ff_factors_capm,
    on="date",
    how="inner"
)

capm_dataset["stock_excess_return"] = capm_dataset["stock_return"] - capm_dataset["rf"]

capm_dataset = capm_dataset[[
    "date",
    "ticker",
    "stock_return",
    "rf",
    "mkt_rf",
    "stock_excess_return"
]].copy()

capm_dataset = capm_dataset.sort_values(["ticker", "date"]).reset_index(drop=True)

save_csv(capm_dataset, capm_dataset_path)

print("CAPM dataset date range:", capm_dataset["date"].min(), "to", capm_dataset["date"].max())
capm_dataset.head()


Saved: /content/ba870_final_project/data/need for app/capm_dataset.csv | shape=(4608, 6)
CAPM dataset date range: 2013-01-31 00:00:00 to 2025-12-31 00:00:00


,date,ticker,stock_return,rf,mkt_rf,stock_excess_return
0,2013-01-31,A,NaN,0.0,0.0557,NaN
1,2013-02-28,A,-0.073693,0.0,0.0128,-0.073693
2,2013-03-31,A,0.014705,0.0,0.0404,0.014705
3,2013-04-30,A,-0.012628,0.0,0.0156,-0.012628
4,2013-05-31,A,0.096766,0.0,0.0280,0.096766


## 11. CAPM risk metrics

This creates `capm_risk_metrics.csv`.


In [ ]:
capm_data = pd.read_csv(capm_dataset_path)
capm_data["date"] = pd.to_datetime(capm_data["date"])

required_cols = [
    "date",
    "ticker",
    "stock_return",
    "rf",
    "mkt_rf",
    "stock_excess_return"
]

missing_cols = [col for col in required_cols if col not in capm_data.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

capm_data = capm_data[required_cols].copy()
capm_data = capm_data.sort_values(["ticker", "date"]).reset_index(drop=True)

capm_data = capm_data.dropna(subset=["ticker", "date", "mkt_rf", "stock_excess_return"]).copy()

obs_count = capm_data.groupby("ticker").size().reset_index(name="n_months")

min_months_required = 24

eligible_tickers = obs_count.loc[
    obs_count["n_months"] >= min_months_required,
    "ticker"
].tolist()

excluded_tickers = obs_count.loc[
    obs_count["n_months"] < min_months_required,
    "ticker"
].copy()

capm_data_filtered = capm_data[capm_data["ticker"].isin(eligible_tickers)].copy()

results = []

for ticker, df_ticker in capm_data_filtered.groupby("ticker"):
    df_ticker = df_ticker.sort_values("date").copy()

    X = sm.add_constant(df_ticker["mkt_rf"])
    y = df_ticker["stock_excess_return"]

    model = sm.OLS(y, X).fit()

    results.append({
        "ticker": ticker,
        "n_months": len(df_ticker),
        "alpha": model.params["const"],
        "beta": model.params["mkt_rf"],
        "r_squared": model.rsquared
    })

capm_risk_metrics = pd.DataFrame(results)

if capm_risk_metrics.empty:
    raise ValueError("capm_risk_metrics is empty. Check CAPM inputs and date alignment.")

def beta_risk_label(beta):
    if beta < 0.80:
        return "Low Beta"
    elif beta <= 1.20:
        return "Moderate Beta"
    else:
        return "High Beta"

capm_risk_metrics["beta_risk_label"] = capm_risk_metrics["beta"].apply(beta_risk_label)

capm_risk_metrics = capm_risk_metrics[[
    "ticker",
    "n_months",
    "alpha",
    "beta",
    "r_squared",
    "beta_risk_label"
]].copy()

capm_risk_metrics = capm_risk_metrics.sort_values("ticker").reset_index(drop=True)

save_csv(capm_risk_metrics, capm_risk_metrics_path)

# Optional diagnostic output
save_csv(excluded_tickers.to_frame(name="ticker"), DATA_DIR / "capm_excluded_tickers.csv")

print("CAPM risk metrics shape:", capm_risk_metrics.shape)
print("Risk label distribution:")
print(capm_risk_metrics["beta_risk_label"].value_counts())
capm_risk_metrics.head()


Saved: /content/ba870_final_project/data/capm_risk_metrics.csv | shape=(30, 6)
Saved: /content/ba870_final_project/data/capm_excluded_tickers.csv | shape=(0, 1)
CAPM risk metrics shape: (30, 6)
Risk label distribution:
beta_risk_label
Low Beta         17
Moderate Beta    10
High Beta         3
Name: count, dtype: int64


,ticker,n_months,alpha,beta,r_squared,beta_risk_label
0,A,155,-0.001074,1.134366,0.467330,Moderate Beta
1,ABBV,155,0.008945,0.689332,0.164577,Low Beta
2,ABT,155,0.001658,0.799330,0.359533,Low Beta
3,ALGN,155,-0.000880,1.713130,0.324527,High Beta
4,AMGN,155,0.004994,0.641009,0.160980,Low Beta


## 12A. Package generated CSV outputs

In [ ]:
# OUTPUT PACKAGE
# This creates one zip file with all regenerated CSVs for easy download from Colab.

import zipfile

zip_path = BASE_DIR / 'ba870_generated_outputs.zip'
files_to_zip = [
    market_raw_path,
    market_monthly_raw_path,
    panel_path,
    model_data_path,
    peer_summary_path,
    ff_clean_path,
    capm_dataset_path,
    capm_risk_metrics_path,
    valuation_comparison_path,
    valuation_predictions_path,
    valuation_coefficients_path,
    ticker_history_path,
]

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in files_to_zip:
        path = Path(path)
        if path.exists():
            z.write(path, arcname=path.name)

print('Created:', zip_path)
print('Download this file from the Colab file browser if needed.')


Created: /content/ba870_final_project/ba870_generated_outputs.zip
Download this file from the Colab file browser if needed.


## 12. Final output audit

This checks that all important CSV outputs were created successfully.


In [ ]:
expected_outputs = [
    market_raw_path,
    market_monthly_raw_path,
    panel_path,
    model_data_path,
    ff_clean_path,
    capm_dataset_path,
    capm_risk_metrics_path,
    peer_summary_path,
    valuation_comparison_path,
    valuation_predictions_path,
    valuation_coefficients_path,
    ticker_history_path
]

audit = pd.DataFrame([audit_csv(path) for path in expected_outputs])
audit


,file,exists,rows,cols
0,/content/ba870_final_project/data/raw/market_d...,True,385,8
1,/content/ba870_final_project/data/raw/market_d...,True,4608,7
2,/content/ba870_final_project/data/panel_data.csv,True,384,25
3,/content/ba870_final_project/data/model_data.csv,True,350,22
4,/content/ba870_final_project/data/ff_factors_c...,True,1196,5
5,/content/ba870_final_project/data/need for app...,True,4608,6
6,/content/ba870_final_project/data/capm_risk_me...,True,30,6
7,/content/ba870_final_project/data/peer_summary...,True,350,29
8,/content/ba870_final_project/data/valuation_mo...,True,4,5
9,/content/ba870_final_project/data/valuation_te...,True,149,9


In [ ]:
missing_or_empty = audit[(audit["exists"] == False) | (audit["rows"].fillna(0) == 0)]

if missing_or_empty.empty:
    print("All expected output files exist and are non-empty.")
else:
    print("Some expected output files are missing or empty:")
    print(missing_or_empty)


All expected output files exist and are non-empty.


## Appendix: Backup Access

If any issues arise when running this notebook, the full project materials can be accessed here:

- Google Drive (all data, notebooks, outputs): https://drive.google.com/drive/folders/1jEMPqKorz-MKobzPQPJ9-Ho3n-9VvC8w?usp=sharing
- Key notebooks:
  - Valuation Model: https://colab.research.google.com/drive/1guNgBF-qxhBUI2Wmsnt1IOhsVttwK-bX?usp=sharing
  - CAPM Risk Model: https://colab.research.google.com/drive/17YFaQ8RoPRIzNXOfLidl2GadIr5uFmFo?usp=sharing